<span style='font-family:Georgia;  font-size: 50px'>  Курс "Пространственный анализ и моделирование в 
     Python" </span>   

<div> <p id="header-bg-title"><br> <br></p> </div>


<span style='font-family:Georgia;font-size: 30px;  '> Блок 6. Создание витрин с пространственными показателями</span>  


<div> <p id="header-bg-title"><br> <br></p> </div>


<span style="font-family:Georgia; font-weight: bold; font-size: 14px "> 
    Автор: Инесса Трегубова <br>
    Канал: @datainthecity
</span>

</span>

<span style="color:red; font-family:Georgia; font-weight: bold; "> ДЗ. В этом ноутбуке  домашних заданий: </span>

- [Задание 1](#task1)
- [Задание 2](#task2)
- [Задание 3](#task3)
- [Задание 4](#task4)

**Навыки:**
   1. Группировать, фильтровать наборы данных по ключевым полям
   2. Объединять слои данных в один набор, используя ключевые поля и пространственные отношения
   3. Рисовать карты в Python

**Цель:**
    научиться создавать дашборды с описанием локаций и создавать на их основе карты в Python

**Когда нужно?**
   1. Когда нужно сравнить несколько локаций и выбрать лучшую по заданным критериям.
   2. Для оценки текущего покрытия сервисом и конкурентами клиентов
   3. Для выявления потребностей территории при создании мастер-плана или плана управления 
   4. При создании дашборда с описанием ключевых метрик для лиц принимающих решения

#### <span style='font-family:Georgia;  font-size: 20px'> Описание задачи </span>  

<span style='font-family:Georgia;'> **Ситуация** </span>   
<span style='font-family:Georgia;'> 
В Лондоне на рынке быстрой доставки еды работают 3 компании. Есть датасет с информацией по ресторанам, с которыми каждый из сервисов сотрудничает. </span> 

<span style='font-family:Georgia;'> **Задача** </span>   
<span style='font-family:Georgia;'> 
Cобрать и представить показатели сервиса в каждом районе (borough) города на карте
             </span>  

## <span style="font-family:Georgia"> Оглавление: </span>
   1. [Агрегация данных на уровне объектов набора](#agg)
   
       1.1 [Анализ содержания набора данных](#profile)
       
       1.2 [Группировка информации по региональным единицам](#groupby)
       <br>
   2. [Визуализация данных на карте](#map)
   
       2.1 [Добавление геометрии в датасет и визуализация методом explore](#geoborough)
       
       2.2 [Создание Choropleth map  с помощью библиотек Matplotlib и Contextily](#contextily)
       
       2.3 [Создание Choropleth map с помощью библиотеки Folium](#folium)
         <br> 
       
       

      

In [34]:
%matplotlib inline

In [35]:
import pandas as pd

from shapely.geometry import Point, Polygon

import numpy as np
import geopandas as gpd

from matplotlib import pyplot as plt

#### <span style='font-family:Georgia;  font-size: 20px'>  Алгоритм для составления описания локации: </span>
1. Определиться, что является ключом, по которому группируется информация
2. Сцепить наборы данных со всеми агрегатами по ключам или с помощью пространственной сцепки
3. Агрегировать значения до ключевых полей с помощью статистических функций
4. Отнормировать данные
5. Обработать выбросы и заполнить пропуски
6. Построить графики для визуализации и сравнения


<a id="agg"> </a>
# <span style='font-family:Georgia;  font-size: 30px'>Агрегация данных на уровне объектов набора</span>

<a id="profile"> </a>
## <span style='font-family:Georgia;background-color:yellow;'> Анализ содержания набора данных  </span>

**1ый этап.** Считываем данные, смотрим,  какие значения содержат поля набора, определяем ключевое поле, которое содержит исследуемые объекты

<span style='font-family:Georgia;'> **Считываем данные** </span >

In [36]:
df_rests = pd.read_excel(
    "data/london_fast_delivery.xlsx",
    sheet_name=0,
    usecols=np.arange(1, 9, 1),
    #
)

print(df_rests.shape)
df_rests.info(memory_usage="deep")
df_rests.nunique()

(6356, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6356 entries, 0 to 6355
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competitor              6356 non-null   object 
 1   latitude                6356 non-null   float64
 2   longitude               6356 non-null   float64
 3   borough                 6356 non-null   object 
 4   cuisine_classification  6356 non-null   object 
 5   price_category          6356 non-null   object 
 6   delivery_radius         6356 non-null   float64
 7   restaurant_type         6356 non-null   object 
dtypes: float64(3), object(5)
memory usage: 2.3 MB


competitor                   3
latitude                  5568
longitude                 5581
borough                     32
cuisine_classification      10
price_category               4
delivery_radius            800
restaurant_type              4
dtype: int64

In [37]:
# В случае ошибки ImportError в ячейке выше расскомментируйте и запустите код в этой ячейке, затем попробуйте еще раз
# ! pip install openpyxl

In [38]:
df_rests.head()

,competitor,latitude,longitude,borough,cuisine_classification,price_category,delivery_radius,restaurant_type
0,Funky Food Co,51.484635,0.103459,Greenwich,Thai,££,4300.000000,Independent
1,Nutrirunners,51.489820,0.120864,Greenwich,Indian,££,3225.000000,Independent
2,Deliveroo,51.508536,-0.277662,Ealing,American,££,3208.333333,Independent
3,Funky Food Co,51.509109,-0.269047,Ealing,American,££,3208.333333,Independent
4,Nutrirunners,51.519071,-0.265791,Ealing,American,£,2406.250000,Independent


<span style='font-family:Georgia;'> **Создаем отчет о содержании датасета** </span >

In [39]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_rests, pool_size=1, progress_bar=False)
profile.to_file("profile.html")

100%|██████████| 8/8 [00:00<00:00, 140.98it/s]


В случае ошибки при отображении в ноутбуке сохраните в html

In [43]:
df_rests.drop_duplicates().shape == df_rests.shape

True

In [44]:
df_rests = df_rests.drop_duplicates()

<a id="task1"> </a>
<span style="color:red; font-family:Georgia; font-weight: bold; ">Задание 1  </span>   
<span style=" font-family:Georgia; font-weight: bold; "> Изучив отчет, ответьте на следующие вопросы и вставьте ответ в тест</span>   

<span style=" font-family:Georgia; font-weight: bold; "> 1. Какой средний радиус доставки?

3489.0786

<span style=" font-family:Georgia; font-weight: bold; " > 2. В каком borough находится самое большое число ресторанов?

Westminster

<span style=" font-family:Georgia; font-weight: bold; " > 3. Изучите раздел interactions и скажите, в рамках какого диапазона широты и долготы находится самое большое число ресторанов?

lon -0.1 to -0.2, lat 51.5 to 51.55

<span style="color:red; font-family:Georgia; font-weight: bold; ">Конец задания   </span>  

<a id='groupby'></a>
## <span style='font-family:Georgia;background-color:yellow;'>Группировка информации по региональным единицам</span>  



<span > Показатели, которые можем посчитать:
 - кол-во ресторанов
 - средняя ценовая категория
 - средний радиус доставки
 - разнообразие кухни </span>

### <span style='font-family:Georgia;'> **Группировка числовых показателей**</span>

In [45]:
df_rests.head()

,competitor,latitude,longitude,borough,cuisine_classification,price_category,delivery_radius,restaurant_type
0,Funky Food Co,51.484635,0.103459,Greenwich,Thai,££,4300.000000,Independent
1,Nutrirunners,51.489820,0.120864,Greenwich,Indian,££,3225.000000,Independent
2,Deliveroo,51.508536,-0.277662,Ealing,American,££,3208.333333,Independent
3,Funky Food Co,51.509109,-0.269047,Ealing,American,££,3208.333333,Independent
4,Nutrirunners,51.519071,-0.265791,Ealing,American,£,2406.250000,Independent


In [46]:
df_rests["rest_id"] = df_rests.index

In [47]:
#заменяем знаки фунта на порядковое значение: 1,2,3
df_rests["price"] = df_rests["price_category"].apply(lambda x: len(x))

In [48]:
df_rests["price"].value_counts()

price
2    4814
1    1280
3     213
4       3
Name: count, dtype: int64

In [49]:
# считаем количество ресторанов, средний радиус доставки и медианную цену для каждого бороу
df_borough = df_rests.groupby("borough")[["delivery_radius", "price", "rest_id"]].agg(
    {"rest_id": "count", "delivery_radius": "mean", "price": "median"}
)

In [50]:
df_borough.head()

,rest_id,delivery_radius,price
borough,,,
Barking and Dagenham,44,3279.640152,2.0
Barnet,195,4173.705408,2.0
Bexley,71,4389.856640,2.0
Brent,228,3543.465378,2.0
Bromley,124,3741.505290,2.0


Переименуем колонку rest_id в соответствии с ее значением

In [51]:
df_borough.rename(columns={"rest_id": "number_rests"}, inplace=True)

### <span style='font-family:Georgia;'> **Группировка категориальных показателей** </span>

<span style='font-family:Georgia;'> **1-ый способ. Pivot table** </span>

In [52]:
df_rests.groupby(["borough", "cuisine_classification"]).size()

borough               cuisine_classification
Barking and Dagenham  American                   7
                      British                    2
                      Chinese                    8
                      Indian                     6
                      Italian                    6
                                                ..
Westminster           Indian                    51
                      Italian                   76
                      Japanese                  55
                      Thai                      16
                      Turkish                   41
Length: 318, dtype: int64

In [53]:
# проверяем есть ли в данных бороу где только один тип кухни
df_rests.groupby(["borough"])["cuisine_classification"].nunique()[
    df_rests.groupby(["borough"])["cuisine_classification"].nunique() == 1
]

Series([], Name: cuisine_classification, dtype: int64)

In [54]:
df_rests.rename(columns={"rest_id": "number_rests"}, inplace=True)

In [55]:
# Создаёт сводную таблицу: строки — районы, столбцы — типы кухни, значения — количество ресторанов
# Считает, сколько ресторанов каждого типа находится в каждом районе
# Это нужно для анализа разнообразия кухонь по районам
df_rests_diversity = pd.pivot_table(
    df_rests,
    index="borough",
    columns="cuisine_classification",
    values="number_rests",
    aggfunc="count",
)

Обработка пропусков

In [56]:
df_rests_diversity.head()

cuisine_classification,American,British,Cafe,Chinese,Dessert,Indian,Italian,Japanese,Thai,Turkish
borough,,,,,,,,,,
Barking and Dagenham,7.0,2.0,NaN,8.0,NaN,6.0,6.0,4.0,4.0,7.0
Barnet,40.0,18.0,13.0,29.0,9.0,23.0,32.0,14.0,6.0,11.0
Bexley,16.0,5.0,2.0,10.0,8.0,13.0,3.0,4.0,4.0,6.0
Brent,43.0,13.0,26.0,35.0,9.0,24.0,16.0,27.0,12.0,23.0
Bromley,15.0,18.0,9.0,17.0,1.0,22.0,13.0,15.0,6.0,8.0


In [57]:
df_rests_diversity.isnull().sum()

cuisine_classification
American    0
British     0
Cafe        1
Chinese     0
Dessert     1
Indian      0
Italian     0
Japanese    0
Thai        0
Turkish     0
dtype: int64

In [58]:
df_rests_diversity.fillna(0, inplace=True)

In [59]:
df_borough.shape

(32, 3)

In [60]:
df_rests_diversity.shape

(32, 10)

<span style='font-family:Georgia;'> **2-ой способ. Get dummies** </span>

In [27]:
# допишите из лекции(df_rests['cuisine_classification']).head()

In [28]:
df_rests_cuisine = df_rests[["borough"]].join(#допишите из лекции(df_rests['cuisine_classification']))

SyntaxError: incomplete input (3980321601.py, line 1)

In [ ]:
df_rests_diversity = df_rests_cuisine.groupby("borough").sum()

In [ ]:
df_rests_diversity

<span style='font-family:Georgia;'> **Сцепка** </span>

In [ ]:
df_borough_full = df_borough.join(df_rests_diversity, how="inner")

In [ ]:
df_borough_full.shape

In [ ]:
df_borough_full.head()

<a id="task2"> </a>
<span style="color:red; font-family:Georgia; font-weight: bold; ">Задание 2  </span>   
<span style=" font-family:Georgia; font-weight: bold; "> Соберите таблицу, где будет показано, сколько в каждом borough ресторанов разного типа (restaurant_type), используя pivot_table,  и сцепите с таблицей df_borough_full. Сохраните результат в таблице df_borough_cousine_type и ответьте на вопросы в тесте </span>   

<span style="color:red; font-family:Georgia; font-weight: bold; ">Конец задания   </span>  

<a id="task3"> </a>
<span style="color:red; font-family:Georgia; font-weight: bold; ">Задание 3  </span>   
<span style=" font-family:Georgia; font-weight: bold; "> Воспользуйтесь фильтрами и выведите районы:  </span>   
  1. где средняя (не медианная!) ценовая категория выше 2
  2. где независимых ресторанов больше, чем ресторанов национальной сети
  3. где количество итальянских ресторанов больше, чем турецких и тайских вместе

<span style="color:red; font-family:Georgia; font-weight: bold; ">Конец задания   </span>  

<a id="map"> </a>
 # <span style='font-family:Georgia;  font-size: 30px'>Визуализация данных на карте  </span>

<a id='geoborough'> </a>
## <span style='font-family:Georgia;background-color:yellow;'> Добавление геометрии в датасет и визуализация методом explore </span>  


1. Для начала нам нужно добавить в датасет геометрию borough

In [ ]:
gdf_borough_geo = gpd.read_file(r"data/london_boroughs.shp")

In [ ]:
gdf_borough_geo.info()

In [ ]:
gdf_borough_geo.head()

In [ ]:
gdf_borough_geo.shape

In [ ]:
gdf_borough_geo.crs

Сцепляем датасеты. Так как в одной таблице borough- индекс, а в другой колонка, то для сцепки делаем оба поля колонками, применяя функцию reset_index() к таблице df_borough_full.     

Сравните

In [ ]:
df_borough_full.head()

In [ ]:
df_borough_full = df_borough_full.reset_index()

In [ ]:
# Объединяет таблицу с данными по районам и геоданными по районам
# Соединение происходит по названию района: "borough" в одной таблице и "Borough" в другой
# Используется inner-join, то есть сохраняются только те районы, которые есть в обеих таблицах
df_borough_full_geo = df_borough_full.merge(
    gdf_borough_geo, how="inner", left_on="borough", right_on="Borough"
)


In [ ]:
df_borough_full_geo.head()

In [ ]:
df_borough_full_geo.shape

**Этапы:**
 - Проверяем тип набора
 - Определяем, что является объектом исследования в  данной таблице
 - Находим/создаем колонку с геометрией, описывающую объект
 - Меняем тип набора на GeoDataFrame, если нужно
 - Определяем целевой CRS, конвертируем, если потребуется

In [ ]:
gdf_borough_full_geo = gpd.GeoDataFrame(
    df_borough_full_geo, crs=gdf_borough_geo.crs, geometry="geometry"
)

In [ ]:
type(gdf_borough_full_geo)

In [ ]:
df_borough_full_geo.shape

In [ ]:
gdf_borough_full_geo  # допишите из лекции

<a id='contextily'></a>
## <span style='font-family:Georgia;background-color:yellow;'> Создание Choropleth map  с помощью библиотек Matplotlib и Contextily</span> 

In [ ]:
import contextily as ctx

from pyproj import CRS

По умолчанию contextily ожидает CRS='epsg:3857' (Сферический Меркатор), поэтому нужно либо указывать CRS внутри метода *add_basemap* или конвертировать датасет в указанную проекцию. Второе будет красивее

In [ ]:
CRS_mercator = CRS.from_string("epsg:3857")

In [ ]:
gdf_borough_full_geo_plot = gdf_borough_full_geo.to_crs(CRS_mercator)

In [ ]:
gdf_borough_full_geo_plot.crs

### <span style='font-family:Georgia'>  Использование непрерывной шкалы для отображения числовых значений </span>

In [ ]:
bounds = gdf_borough_full_geo.total_bounds

In [ ]:
bounds

In [ ]:
# column (в plot) - имя колонки значения которой используются для цветовой палитры
# legend (в plot) - True/False - отображать, не отображать легенду
# source (в add_basemap) - отвечает за подложку карты

f, ax = plt.subplots(1, figsize=(20, 10))
# ax.imshow(london_img, extent=london_ext)
gdf_borough_full_geo.plot(column="number_rests", legend=True, ax=ax)
ctx.add_basemap(ax=ax, crs=gdf_borough_full_geo.crs.to_string())
plt.title(
    "Карта с числом ресторанов с быстрой доставкой по районам Лондона с непрерывной цветовой шкалой"
)

In [ ]:
gdf_borough_full_geo_plot.head()

### <span style='font-family:Georgia'>  Использование интервалов для лучшего визуального сравнения </span>

In [ ]:
f,ax = plt.subplots(1,1, figsize =(20,15))

#column - имя колонки значения которой используются для цветовой палитры
#legend - True/False - отображать, не отображать легенду
#scheme - тип деления на интервалы, если не указано, то непрерывная шкала. Только для choropleth. 
# k - кол-во интервалов в схеме

gdf_borough_full_geo_plot.plot(column="number_rests" ,legend=True,  ax=ax,#допишите из лекции);
ctx.#допишите из лекции, но вместо Stamen.TonerLite используейте CartoDB.Positron 
plt.title('Карта с числом ресторанов с быстрой доставкой по районам Лондона с непрерывной цветовой шкалой');

Если возникла ошибка с **отсутствием mapclassify**  на предыдущем шаге, выполните код ниже (раскомментировав его)

In [ ]:
# ! pip install mapclassify --upgrade
# import importlib
# importlib.reload(mapclassify)

<a id='folium'> </a>
## <span style='font-family:Georgia;background-color:yellow;'> Создание Choropleth map с помощью библиотеки Folium</span> 

О том, как работать с другими типами карт смотрите в дополнительных материалах в ноутбуке Неделя 3. Типы карт с библиотекой Folium

In [ ]:
import folium

### <span style='font-family:Georgia'> Шаг 1. Поиск точки старта - центра карты </span>

In [ ]:
gdf_borough_full_geo.geometry.union_all().centroid #вместо unary_union теперь используйте unary_all()

In [ ]:
point_start = #допишите из лекции вместо unary_union теперь используйте unary_all()

In [ ]:
point_start.xy

Так как мы хотим сравнить 2 способа раскрасить карту, то сразу создаем основы для двух карт

In [ ]:
# zoom_start определяет начальный уровень масштабирования карты:
# чем больше значение, тем ближе "приближение" (детализация) карты.
# Как выбрать zoom_start?
#3–5 — вся страна / весь континент
#10–12 — город (обычно этого достаточно для Москвы, Лондона и т.д.)
#15+ — улицы и здания
#18+ — очень крупный масштаб (уровень дома)

print([point_start.y, point_start.x])
m1 = folium.Map([point_start.y, point_start.x], zoom_start=10)
m2 = folium.Map([point_start.y, point_start.x], zoom_start=10)

### <span style='font-family:Georgia'> Шаг 2.Сохранение датасета в geojson в нужном для folium формате </span>

In [ ]:
file_name = "gdf_borough_full_geo.geojson"
# выбираем только колонки нужные для построения карты
df2geojson = gdf_borough_full_geo[["borough", "geometry", "number_rests"]]
df2geojson.to_file(
    file_name, index=df2geojson.index.tolist(), driver="GeoJSON"
)  # лучше передавать driver, потому что не всегда корректно распознает формат

In [ ]:
df2geojson

### <span style='font-family:Georgia'> Шаг 3. Добавление элементов на карту</span>

**1-ый вариант. Указываем кол-во интервалов цифрой**

In [ ]:
bins_input = 5

In [ ]:
cp = folium.Choropleth(
    geo_data=file_name,  # геоданные в формате GeoJSON (границы районов)
    name="number_rests",  # имя слоя, отображаемое в легенде карты
    data=df2geojson,  # таблица с данными для отображения (район + число ресторанов)
    columns=["borough", "number_rests"],  # какие столбцы использовать: ключ и значение
    bins=bins_input,  # интервалы значений для раскраски (например, 5 групп по количеству ресторанов)
    key_on="feature.properties.borough",  # имя поля в geojson, по которому идёт соединение с таблицей
    fill_color="YlOrRd",  # цветовая схема (от жёлтого к красному)
    fill_opacity=0.5,  # прозрачность заливки
    line_opacity=0,  # прозрачность границ (0 — без границ)
    legend_name="number_rests",  # название легенды на карте
).add_to(m1)  # добавление слоя на карту m1


In [ ]:
m1

**2-ой вариант. Указываем кол-во интервалов с помощью персентилей**

Правила для создания интервалов для количественной переменной:
   1. Лист должен начинаться с минимального значения в столбце, заканчиваться максимальным
   2. Тип данных должен быть float: int python воспринимает как категориальную переменную и пытается уменьшить количество интервалов, что приводит к ошибке.

In [ ]:
bins_qunatiles = np.quantile(
    gdf_borough_full_geo["number_rests"], q=np.arange(0, 1.1, 0.1) # создаем лист из 10 значений с шагом 0.1
)

In [ ]:
bins_interval_qntls = list(bins_qunatiles)

In [ ]:
# bins_interval_qntls = bins_interval_qntls[1:]

In [ ]:
bins_interval_qntls

In [ ]:
cp1 = folium.Choropleth(
    geo_data=file_name,
    name="number_rests",
    data=df2geojson,
    columns=["borough", "number_rests"],
    bins=bins_interval_qntls,
    key_on="feature.properties.borough",  # имя поля с ключом. Важно, чтобы не было дублей!
    fill_color="YlOrRd",  #
    fill_opacity=0.5,
    line_opacity=0,
    legend_name="number_rests",
).add_to(m2)

### <span style='font-family:Georgia'> Шаг 4. Визуализация </span>

In [ ]:
m1

In [ ]:
m2

 сохраним шаги в функцию, чтобы не повторять их каждый раз. Воспользуйтесь ей в задании 4

In [ ]:
def visual_choropleth(
    my_df,
    value,
    key,
    geometry,
    m,
    name,
    bins_num=5,
    bins_interval=[],
    fill_color="YlOrRd",
):
    df = my_df[[key, geometry, value]].copy()

    file_name = name + ".geojson"
    df.to_file(file_name)

    if len(bins_interval) == 0:
        bins_input = bins_num
    else:
        bins_input = bins_interval
    print(bins_input)
    cp = folium.Choropleth(
        geo_data=file_name,
        name=value,
        data=df,
        columns=[key, value],
        bins=bins_input,
        key_on=f"feature.properties.{key}",
        fill_color=fill_color,  #
        fill_opacity=0.5,
        line_opacity=0,
        legend_name=name,
    ).add_to(m)
    return cp

<a id='task4'></a>
<span style="color:red; font-family:Georgia; font-weight: bold; "> Задание 4 </span>         

<span style=" font-family:Georgia; font-weight: bold; "> Нарисуйте choropleth map для среднего радиуса доставки в районах Лондона, используя библиотеки folium и matplotlib.  </span>     

<span style=" font-family:Georgia; font-weight: bold; ">
Для folium поменяйте цветовую палитру на любую вам понравившуюся и сделайте 7 интервалов с (почти) равным числом наблюдений внутри. Для matplotlib поменяйте схему на JenksBreaks(natural breaks) и число интервалов =7. Какая из карт на ваш взгляд лучше позволяет выделить районы с самым большим радиусом доставки?
</span>   

В matplotlib поменяйте подлоджку на одну из тех, что описаны тут - https://contextily.readthedocs.io/en/latest/providers_deepdive.html

Названия палитр для Folium есть, например, тут  - https://user-images.githubusercontent.com/17128994/115975254-c2031e00-a56b-11eb-8025-d82d36bfda1d.png

<span style=" font-family:Georgia; font-weight: bold; "> Matplotlib</span>

<span style=" font-family:Georgia; font-weight: bold; "> Folium</span>

<span style="color:red; font-family:Georgia; font-weight: bold; ">Конец задания  </span>